# Well model QC (standard template)

Run by `opmagent` for every well model version built from well tables; the HTML export is the version's QC report and the approval is tied to its sha256. **Read only:** the notebook loads the build payload and the measured history (`history.csv`), the well tables from the database and, when the version names a geo version, that grid (perforations are placed in it as the deck builder does). It computes checks and plots and writes `qc_summary.json`. Don't edit this template for one model: write an investigation notebook instead (`opmagent well investigate`).

In [ ]:
payload_path = "payload.json"   # written by the well pipeline
db_path = ""                    # SQLite database (opened read-only)
version_id = 0                  # well_model_versions.id
summary_path = "qc_summary.json"

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'   # self-contained HTML export
import pandas as pd
from IPython.display import display
from opmagent.notebooks import well_qc_lib as qc

data = qc.load(payload_path, db_path, version_id)
findings = qc.run_all_checks(data)
qc.header(data)

## Summary

Every QC finding, worst first. Fail and warn findings don't block the build; they are for the reviewer.

In [ ]:
qc.show_findings(findings)

## Assumptions, input gaps and validator warnings

In [ ]:
display(qc.show_list(data.get('assumptions', []), title='Assumptions'))
display(qc.show_list([f"{g['severity']}: {g['item']}: {g['message']}" for g in data.get('gaps', [])], title='Input gaps'))
qc.show_list(title='Validator warnings', items=[w['message'] + (f" ({w['where']})" if w.get('where') else '') for w in data.get('validator_warnings', [])])

## 1. Wells and trajectories

Well heads against the first survey station, the wells against the geo grid, and the surveys themselves: a step whose straight 3D distance exceeds its MD step is impossible, and a high dogleg severity is worth a second look.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] in ('wells', 'trajectories')]))
display(data['wells'].drop(columns='version_id'))
if len(data['trajectory']):
    display(qc.fig_plan(data))

## 2. Perforations

Perforations need to be open before the well's first open control (Flow shuts a well with no open connection). With a geo version, each MD interval is sampled every 0.5 m and placed in the grid's cells, as the deck builder does: how much lies outside the grid's layers, how tight the perforated cells are, and whether producers are perforated below the OWC or above the GOC.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] in ('completions',)]))
p = qc.place(data)
if len(p):
    f = qc.fig_perforated_cells(data)
    if f is not None:
        display(f)
    display(p.round(3))
else:
    display(data['completions'].drop(columns='version_id'))

In [ ]:
for w in list(data['wells']['well_name'])[:12]:   # the first 12 wells
    if (data['trajectory']['well_name'] == w).any():
        display(qc.fig_section(data, w))

## 3. History

The monthly rates are each month's volume over its producing days. `WCONHIST` applies a rate for the whole month, so the volume the deck applies is compared with the measured volume, per well and phase. Missing months, spikes and missing BHP are listed too.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] in ('history',)]))
f = qc.fig_cumulative(data)
if f is not None:
    display(f)
for w in list(data['wells']['well_name'])[:12]:
    if len(data['events']) and ((data['events']['well_name'] == w) & (data['events']['mode'] == 'HISTORY')).any():
        display(qc.fig_history(data, w))

In [ ]:
v = qc.model_volumes(data)
if len(v):
    cols = [c for c in ('oil_model', 'oil', 'water_model', 'water', 'gas_model', 'gas', 'inj_model', 'water_inj') if c in v]
    display(v.groupby('well_name')[cols].sum().round(0))

## 4. Controls and schedule

Events against the start date, prediction against the end of the history, and every well's open and shut periods.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] in ('controls',)]))
f = qc.fig_schedule(data)
if f is not None:
    display(f)
pred = data['events'][data['events']['mode'] == 'PREDICTION'] if len(data['events']) else data['events']
if len(pred):
    display(pred.drop(columns=['version_id', 'id']).dropna(axis=1, how='all'))

## Result

In [ ]:
summary = qc.write_summary(summary_path, findings, data)
print(f"Overall: {summary['overall'].upper()}  " + ", ".join(f'{k} {v}' for k, v in summary['counts'].items()))